# Tennis 3D Tracking Visualization

Interactive visualization of the full tracking pipeline output:
- 2D pixel detections (cam66/cam68)
- 3D triangulated trajectory
- Bounce detection & net crossings
- Speed profile & court heatmaps

In [ ]:
# Cell 1: Load data + frame range parameters
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ============================================================
# ★ Data file — change this to load different exports
# ============================================================
DATA_FILE = '../exports/tracking_0323_data.json'    # 新视频 (cam66/68_20260323)
# DATA_FILE = '../exports/tracking_2min_data.json'  # 旧视频 (cam66/68_20260307)
# ============================================================

# ============================================================
# ★ Frame range parameters — modify these to zoom in
# ============================================================
FRAME_START = 0       # start frame (inclusive)
FRAME_END   = None    # end frame (exclusive), set to None for all frames
# ============================================================

with open(DATA_FILE) as f:
    data = json.load(f)

meta = data['meta']
court = meta['court']
FPS = meta['fps']

# Parse detections into arrays
def parse_det(d):
    frames = sorted(int(k) for k in d.keys())
    if not frames: return np.array([]), np.array([]), np.array([]), np.array([])
    px = np.array([d[str(f)]['px'] for f in frames])
    py = np.array([d[str(f)]['py'] for f in frames])
    conf = np.array([d[str(f)]['blob_sum'] for f in frames])
    return np.array(frames), px, py, conf

def parse_3d(d):
    frames = sorted(int(k) for k in d.keys())
    if not frames: return np.array([]), np.array([]), np.array([]), np.array([])
    x = np.array([d[str(f)]['x'] for f in frames])
    y = np.array([d[str(f)]['y'] for f in frames])
    z = np.array([d[str(f)].get('z', 0) for f in frames])
    return np.array(frames), x, y, z

def filter_range(frames, *arrays):
    """Filter arrays by FRAME_START/FRAME_END."""
    if len(frames) == 0: return [frames] + list(arrays)
    end = FRAME_END if FRAME_END is not None else int(frames.max()) + 1
    mask = (frames >= FRAME_START) & (frames < end)
    return [frames[mask]] + [a[mask] for a in arrays]

f66, px66, py66, conf66 = parse_det(data['det66'])
f68, px68, py68, conf68 = parse_det(data['det68'])
f3d, x3d, y3d, z3d = parse_3d(data['points_3d'])
fs, xs, ys, zs = parse_3d(data['smoothed_3d'])

# Apply frame range filter
f66, px66, py66, conf66 = filter_range(f66, px66, py66, conf66)
f68, px68, py68, conf68 = filter_range(f68, px68, py68, conf68)
f3d, x3d, y3d, z3d = filter_range(f3d, x3d, y3d, z3d)
fs, xs, ys, zs = filter_range(fs, xs, ys, zs)

# Filter bounces and net crossings by frame range
end_frame = FRAME_END if FRAME_END is not None else 999999
bounces = [b for b in data['bounces'] if FRAME_START <= b['frame'] < end_frame]
net_crossings = [nc for nc in data['net_crossings'] if FRAME_START <= nc['frame'] < end_frame]
speed_data = {k: v for k, v in data['speed_profile'].items() if FRAME_START <= int(k) < end_frame}

print(f'Data file: {DATA_FILE}')
print(f'Total frames: {meta["n_frames"]} @ {FPS} fps')
print(f'Showing frames: {FRAME_START} - {FRAME_END or "end"}')
print(f'  Smoothed 3D points: {len(fs)}')
print(f'  Bounces: {len(bounces)}, Net crossings: {len(net_crossings)}')

In [ ]:
# Cell 2: Overview statistics
print('=' * 60)
print('TRACKING PIPELINE SUMMARY')
print('=' * 60)
print(f'Total frames:         {meta["n_frames"]}')
print(f'Duration:             {meta["n_frames"]/FPS:.1f}s ({meta["n_frames"]/FPS/60:.1f} min)')
print(f'Mode:                 {meta["mode"]}')
print(f'Frame alignment:      {meta.get("frame_alignment", "N/A")}')
print()
print('--- Detection ---')
print(f'cam66 detections:     {len(data["det66"])} ({100*len(data["det66"])/meta["n_frames"]:.1f}%)')
print(f'cam68 detections:     {len(data["det68"])} ({100*len(data["det68"])/meta["n_frames"]:.1f}%)')
print(f'Multi-blob cam66:     {len(data["multi66"])} frames')
print(f'Multi-blob cam68:     {len(data["multi68"])} frames')
print()
print('--- Triangulation ---')
print(f'Raw 3D points:        {len(data["points_3d"])}')
print(f'Smoothed 3D points:   {len(data["smoothed_3d"])}')
if 'viterbi_stats' in meta:
    vs = meta['viterbi_stats']
    print(f'Viterbi segments:     {vs.get("segments", "?")}')
    print(f'Viterbi matched:      {vs.get("matched_frames", "?")}/{vs.get("total_frames", "?")}')
print(f'Stereo upgraded:      {meta.get("stereo_upgraded", "N/A")}')
print()
print('--- Events ---')
print(f'Bounces:              {len(bounces)} (IN={meta["bounces_in_court"]}, OUT={meta["bounces_out"]})')
print(f'Net crossings:        {len(net_crossings)}')
avg_speed = np.mean([nc['speed_kmh'] for nc in net_crossings]) if net_crossings else 0
print(f'Avg crossing speed:   {avg_speed:.0f} km/h')
print()
print('--- Rally ---')
if 'rally_preds' in data:
    rally_preds = np.array(data['rally_preds'])
    print(f'Rally frames:         {int(rally_preds.sum())} ({100*rally_preds.mean():.1f}%)')
else:
    print('Rally segmentation:   not available')
print('=' * 60)

In [ ]:
# Cell 3: 2D pixel trajectories (cam66 and cam68)
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# cam66
sc66 = axes[0].scatter(f66, py66, c=f66, cmap='viridis', s=2, alpha=0.6)
axes[0].set_title('cam66 - Detected Y pixel over time')
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Pixel Y')
axes[0].invert_yaxis()
plt.colorbar(sc66, ax=axes[0], label='Frame')

# cam68
sc68 = axes[1].scatter(f68, py68, c=f68, cmap='viridis', s=2, alpha=0.6)
axes[1].set_title('cam68 - Detected Y pixel over time')
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('Pixel Y')
axes[1].invert_yaxis()
plt.colorbar(sc68, ax=axes[1], label='Frame')

# Mark bounce frames
for b in bounces:
    for ax in axes:
        ax.axvline(b['frame'], color='red', alpha=0.3, linewidth=0.8)
for nc in net_crossings:
    for ax in axes:
        ax.axvline(nc['frame'], color='blue', alpha=0.2, linewidth=0.8)

plt.tight_layout()
plt.show()

# X-Y scatter for both cameras
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(px66, py66, c=f66, cmap='viridis', s=2, alpha=0.5)
axes[0].set_title('cam66 - Pixel XY')
axes[0].set_xlabel('Pixel X')
axes[0].set_ylabel('Pixel Y')
axes[0].invert_yaxis()
axes[0].set_xlim(0, 1920)
axes[0].set_ylim(1080, 0)

axes[1].scatter(px68, py68, c=f68, cmap='viridis', s=2, alpha=0.5)
axes[1].set_title('cam68 - Pixel XY')
axes[1].set_xlabel('Pixel X')
axes[1].set_ylabel('Pixel Y')
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1920)
axes[1].set_ylim(1080, 0)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: 3D trajectory (plotly interactive)
# ============================================================
VIEW_START = None   # None = follow global FRAME_START
VIEW_END   = None   # None = follow global FRAME_END
# ============================================================

_vs = VIEW_START if VIEW_START is not None else FRAME_START
_ve = VIEW_END if VIEW_END is not None else (FRAME_END or 999999)
_mask_s = (fs >= _vs) & (fs < _ve)
_fs, _xs, _ys, _zs = fs[_mask_s], xs[_mask_s], ys[_mask_s], zs[_mask_s]
_bounces = [b for b in bounces if _vs <= b['frame'] < _ve]
_ncs = [nc for nc in net_crossings if _vs <= nc['frame'] < _ve]

# V2 coords: origin at center, net at y=0
HW = court.get('half_width', court.get('singles_x_max', 2.745))
HL = court.get('half_length', court.get('court_length', 23.77) / 2)
# Auto-detect: if singles_x_min > 0, old coords; else V2
if court.get('singles_x_min', 0) > 0:  # old coordinate system
    HW = (court['singles_x_max'] - court['singles_x_min']) / 2
    HL = court['court_length'] / 2
NET_Y = court.get('net_y', 0)
NET_H = 1.07
SVC = 6.400

fig = go.Figure()

# Court surface
fig.add_trace(go.Mesh3d(
    x=[-HW, HW, HW, -HW], y=[-HL, -HL, HL, HL], z=[0]*4,
    i=[0,0], j=[1,2], k=[2,3], color='#8B7FBF', opacity=0.8,
    showlegend=False, hoverinfo='skip'))

# Court outline
ol = [[-HW,-HL,0],[HW,-HL,0],[HW,HL,0],[-HW,HL,0],[-HW,-HL,0]]
fig.add_trace(go.Scatter3d(x=[p[0] for p in ol], y=[p[1] for p in ol], z=[0]*5,
    mode='lines', line=dict(color='white', width=10), showlegend=False, hoverinfo='skip'))

# Service lines + center line
fig.add_trace(go.Scatter3d(
    x=[-HW,HW,None,-HW,HW,None,0,0],
    y=[-SVC,-SVC,None,SVC,SVC,None,-SVC,SVC], z=[0]*8,
    mode='lines', line=dict(color='white', width=8), showlegend=False, hoverinfo='skip'))

# Net
fig.add_trace(go.Mesh3d(
    x=[-HW-0.5,HW+0.5,HW+0.5,-HW-0.5], y=[NET_Y]*4, z=[0,0,NET_H,NET_H],
    i=[0,0], j=[1,2], k=[2,3], color='darkgray', opacity=0.4, showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter3d(x=[-HW-0.5,HW+0.5], y=[NET_Y,NET_Y], z=[NET_H,NET_H],
    mode='lines', line=dict(color='black', width=12), showlegend=False, hoverinfo='skip'))

# Trajectory
heights = _zs
fig.add_trace(go.Scatter3d(
    x=_xs, y=_ys, z=_zs, mode='lines',
    line=dict(color=heights, colorscale='Hot', width=3,
              colorbar=dict(title='Height(m)', x=1.02, len=0.5, thickness=15)),
    name='Trajectory', hoverinfo='skip'))

# Bounces
if _bounces:
    for b in _bounces:
        color = 'red' if not b.get('in_court', True) else 'lime'
        fig.add_trace(go.Scatter3d(
            x=[b['x']], y=[b['y']], z=[b['z']], mode='markers+text',
            marker=dict(size=8, color=color, symbol='diamond'),
            text=[f'f{b["frame"]}'], textposition='top center', textfont=dict(size=10, color='white'),
            showlegend=False,
            hovertext=f'Bounce f{b["frame"]}<br>z={b["z"]:.3f}', hoverinfo='text'))

# Moving ball + animation
fig.add_trace(go.Scatter3d(
    x=[_xs[0]] if len(_xs) else [0], y=[_ys[0]] if len(_ys) else [0], z=[_zs[0]] if len(_zs) else [0],
    mode='markers', marker=dict(size=5, color='yellow', line=dict(color='orange', width=2)),
    name='Ball'))

ball_idx = len(fig.data) - 1
n = len(_fs)
dur = max(n/25, 5)
frames_anim = [go.Frame(data=[go.Scatter3d(x=[_xs[i]],y=[_ys[i]],z=[_zs[i]])],
                         name=str(i), traces=[ball_idx]) for i in range(n)]
fig.frames = frames_anim

fig.update_layout(
    title=f'3D Trajectory (frames {_vs}-{_ve if _ve<999999 else "end"})',
    scene=dict(
        xaxis=dict(title='X (m)', range=[-HW-2, HW+2], backgroundcolor='#e0f2fe'),
        yaxis=dict(title='Y (m)', range=[-HL-3, HL+3], backgroundcolor='#e0f2fe'),
        zaxis=dict(title='Z (m)', range=[-0.5, 6], backgroundcolor='#dbeafe'),
        aspectmode='data', camera=dict(eye=dict(x=1.8, y=-1.8, z=1.2)),
        bgcolor='#f0f9ff'),
    updatemenus=[dict(type='buttons', buttons=[
        dict(label='Play', method='animate', args=[None, {'frame':{'duration':1000/max(n/dur,1),'redraw':False},'fromcurrent':True}]),
        dict(label='Pause', method='animate', args=[[None],{'frame':{'duration':0,'redraw':False},'mode':'immediate'}])],
        x=0.12, y=0.02)],
    sliders=[{'steps':[{'args':[[f.name],{'frame':{'duration':0},'mode':'immediate'}],
              'label':str(k) if k%50==0 else '','method':'animate'} for k,f in enumerate(frames_anim)],
              'x':0.12,'len':0.75,'currentvalue':{'prefix':'Frame: '}}] if frames_anim else [],
    width=1200, height=800)
fig.show()

In [ ]:
# Cell 5: Z-axis over time
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(f3d, z3d, '.', color='lightgray', markersize=1, alpha=0.4, label='Raw 3D')
ax.plot(fs, zs, '-', color='blue', linewidth=0.8, alpha=0.8, label='Smoothed')

for b in bounces:
    color = 'green' if b['in_court'] else 'red'
    ax.axvline(b['frame'], color=color, alpha=0.5, linewidth=1.5, linestyle='--')
    ax.plot(b['frame'], b['z'], 'v', color=color, markersize=10)
    ax.annotate(f'f{b["frame"]}', (b['frame'], b['z']),
                textcoords='offset points', xytext=(5, 10), fontsize=8, color=color)

for nc in net_crossings:
    ax.axvline(nc['frame'], color='cyan', alpha=0.3, linewidth=1)

ax.axhline(0.35, color='orange', alpha=0.3, linestyle=':', label='Near-end z threshold')
ax.axhline(0.65, color='red', alpha=0.3, linestyle=':', label='Far-end z threshold')

ax.set_xlabel('Frame')
ax.set_ylabel('Z (meters)')
ax.set_title(f'Ball Height (Z) Over Time (frames {FRAME_START}-{FRAME_END or "end"})')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(-0.5, max(zs.max() if len(zs) > 0 else 3, 3) + 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Speed over time
sp_frames = sorted(int(k) for k in speed_data.keys())
sp_kmh = np.array([speed_data[str(f)]['speed_kmh'] for f in sp_frames])

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(sp_frames, sp_kmh, '-', color='purple', linewidth=0.8, alpha=0.7, label='Ball speed')

# Highlight net crossing speeds
for nc in net_crossings:
    ax.plot(nc['frame'], nc['speed_kmh'], 'o', color='cyan', markersize=8, zorder=5)
    ax.annotate(f'{nc["speed_kmh"]:.0f}', (nc['frame'], nc['speed_kmh']),
                textcoords='offset points', xytext=(5, 8), fontsize=7, color='darkblue')

# Mark bounces
for b in bounces:
    ax.axvline(b['frame'], color='red', alpha=0.3, linewidth=1)

ax.set_xlabel('Frame')
ax.set_ylabel('Speed (km/h)')
ax.set_title('Ball Speed Over Time (3D)')
ax.legend()
ax.set_ylim(0, min(sp_kmh.max() + 20 if len(sp_kmh) > 0 else 200, 300))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Speed histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sp_kmh[sp_kmh < 250], bins=50, color='purple', alpha=0.7)
ax.set_xlabel('Speed (km/h)')
ax.set_ylabel('Count')
ax.set_title('Speed Distribution')
nc_speeds = [nc['speed_kmh'] for nc in net_crossings]
if nc_speeds:
    for s in nc_speeds:
        ax.axvline(s, color='cyan', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: XY scatter (bird's eye view) — V2 coordinate system
def draw_court_matplotlib(ax):
    """Draw singles court. Auto-detects V2 (center origin) or V1 coords."""
    # V2: half_width/half_length or detect from singles_x_min
    hw = court.get('half_width', None)
    hl = court.get('half_length', None)
    net_y = court.get('net_y', 0)
    
    if hw is None:  # old coord system
        sx_min = court.get('singles_x_min', 1.37)
        sx_max = court.get('singles_x_max', 6.86)
        cl = court.get('court_length', 23.77)
        hw = (sx_max - sx_min) / 2
        hl = cl / 2
    
    svc = 6.400
    
    rect = patches.Rectangle((-hw, -hl), 2*hw, 2*hl,
                              linewidth=2, edgecolor='white', facecolor='#2d5e1e', zorder=0)
    ax.add_patch(rect)
    # Service lines
    for sy in [-svc, svc]:
        ax.plot([-hw, hw], [sy, sy], 'w-', linewidth=1)
    ax.plot([0, 0], [-svc, svc], 'w-', linewidth=1)  # center service line
    # Net
    ax.plot([-hw-0.5, hw+0.5], [net_y, net_y], 'w-', linewidth=3, alpha=0.8)
    # Baselines
    for y in [-hl, hl]:
        ax.plot([-hw, hw], [y, y], 'w-', linewidth=2)
    for x in [-hw, hw]:
        ax.plot([x, x], [-hl, hl], 'w-', linewidth=2)
    # Center marks
    ax.plot([0, 0], [-hl, -hl+0.3], 'w-', linewidth=1)
    ax.plot([0, 0], [hl-0.3, hl], 'w-', linewidth=1)
    
    ax.set_facecolor('#1a3a0f')
    ax.set_xlim(-hw - 2, hw + 2)
    ax.set_ylim(-hl - 2, hl + 2)
    ax.set_aspect('equal')

fig, ax = plt.subplots(figsize=(6, 14))
draw_court_matplotlib(ax)

sc = ax.scatter(xs, ys, c=fs, cmap='plasma', s=4, alpha=0.6, zorder=2)
plt.colorbar(sc, ax=ax, label='Frame', shrink=0.5)

for b in bounces:
    color = 'lime' if b['in_court'] else 'red'
    bx = b.get('x_homo', b['x'])
    by = b.get('y_homo', b['y'])
    ax.plot(bx, by, 'o', color=color, markersize=12, markeredgecolor='white',
            markeredgewidth=1.5, zorder=5)
    ax.annotate(f'f{b["frame"]}', (bx, by), color='white', fontsize=7,
                ha='center', va='center', fontweight='bold', zorder=6)

ax.set_xlabel('X (meters)')
ax.set_ylabel('Y (meters)')
ax.set_title(f'Bird\'s Eye View (frames {FRAME_START}-{FRAME_END or "end"})')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Bounce analysis — labeled with frame number
smoothed_lookup = data['smoothed_3d']

n_bounces = len(bounces)
if n_bounces == 0:
    print('No bounces detected in this frame range.')
else:
    cols = min(4, n_bounces)
    rows = (n_bounces + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), squeeze=False)
    
    for idx, b in enumerate(bounces):
        ax = axes[idx // cols][idx % cols]
        fi = b['frame']
        
        window_frames = []
        window_z = []
        for offset in range(-10, 11):
            f_check = fi + offset
            sf = str(f_check)
            if sf in smoothed_lookup:
                window_frames.append(f_check)
                window_z.append(smoothed_lookup[sf]['z'])
        
        if window_frames:
            ax.plot(window_frames, window_z, 'b.-', linewidth=1.5, markersize=4)
            ax.axvline(fi, color='red', alpha=0.5, linestyle='--')
            ax.plot(fi, b['z'], 'rv', markersize=12)
        
        tag = 'IN' if b['in_court'] else 'OUT'
        cam = b.get('cam_used', '?')
        ax.set_title(f'Frame {fi} ({tag})\n'
                     f'z={b["z"]:.3f}m, cam={cam}', fontsize=9)
        ax.set_xlabel('Frame')
        ax.set_ylabel('Z (m)')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.2, max(window_z + [1.0]) + 0.3 if window_z else 1.5)
    
    for idx in range(n_bounces, rows * cols):
        axes[idx // cols][idx % cols].set_visible(False)
    
    plt.suptitle('Bounce Z-Profile (V-shape Analysis)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print bounce table with frame numbers
    print(f'{"Frame":>7} {"X":>7} {"Y":>7} {"Z":>6} {"In/Out":>6} {"Camera":>8}')
    print('-' * 50)
    for b in bounces:
        bx = b.get('x_homo', b['x'])
        by = b.get('y_homo', b['y'])
        tag = 'IN' if b['in_court'] else 'OUT'
        cam = b.get('cam_used', '3d')
        print(f'{b["frame"]:>7} {bx:>7.2f} {by:>7.2f} {b["z"]:>6.3f} {tag:>6} {cam:>8}')

In [ ]:
# Cell 9: Detection confidence over time
fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

axes[0].plot(f66, conf66, '.', color='orange', markersize=1, alpha=0.5)
axes[0].set_ylabel('blob_sum')
axes[0].set_title('cam66 Detection Confidence (blob_sum)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(f68, conf68, '.', color='teal', markersize=1, alpha=0.5)
axes[1].set_ylabel('blob_sum')
axes[1].set_xlabel('Frame')
axes[1].set_title('cam68 Detection Confidence (blob_sum)')
axes[1].grid(True, alpha=0.3)

# Mark bounce and net crossing frames
for b in bounces:
    for ax in axes:
        ax.axvline(b['frame'], color='red', alpha=0.3, linewidth=0.8)
for nc in net_crossings:
    for ax in axes:
        ax.axvline(nc['frame'], color='blue', alpha=0.2, linewidth=0.8)

# Rally overlay
if 'rally_preds' in data:
    rp = np.array(data['rally_preds'])
    rally_on = np.where(np.diff(rp.astype(int)) == 1)[0]
    rally_off = np.where(np.diff(rp.astype(int)) == -1)[0]
    for ax in axes:
        for start in rally_on:
            # Find corresponding end
            ends = rally_off[rally_off > start]
            end = ends[0] if len(ends) > 0 else len(rp) - 1
            ax.axvspan(start, end, alpha=0.08, color='green')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Heatmap of ball positions on court — V2 coordinates
hw = court.get('half_width', (court.get('singles_x_max',6.86)-court.get('singles_x_min',1.37))/2)
hl = court.get('half_length', court.get('court_length',23.77)/2)
net_y = court.get('net_y', 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 14))

# Left: smoothed trajectory heatmap
ax = axes[0]
draw_court_matplotlib(ax)
h, xedges, yedges = np.histogram2d(
    xs, ys, bins=[30, 60],
    range=[[-hw-1, hw+1], [-hl-1, hl+1]])
h = h.T
h_masked = np.ma.masked_where(h == 0, h)
ax.pcolormesh(xedges, yedges, h_masked, cmap='hot', alpha=0.6, zorder=1)
ax.set_title('Ball Position Heatmap')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')

# Right: bounce locations
ax = axes[1]
draw_court_matplotlib(ax)
smoothed_lookup = data['smoothed_3d']

for i, b in enumerate(bounces):
    bx = b.get('x_homo', b['x'])
    by = b.get('y_homo', b['y'])
    color = 'lime' if b['in_court'] else 'red'
    ax.plot(bx, by, 'o', color=color, markersize=16, markeredgecolor='white',
            markeredgewidth=2, zorder=5)
    ax.annotate(f'f{b["frame"]}', (bx, by), color='white', fontsize=9,
                ha='center', va='center', fontweight='bold', zorder=6)

# Net crossing arrows
for nc in net_crossings:
    sf = str(nc['frame'])
    if sf in smoothed_lookup:
        pt = smoothed_lookup[sf]
        dy = 0.8 if nc['direction'] == 'near_to_far' else -0.8
        ax.annotate('', xy=(pt['x'], net_y + dy), xytext=(pt['x'], net_y),
                    arrowprops=dict(arrowstyle='->', color='cyan', lw=2), zorder=4)

ax.set_title('Bounce Locations & Net Crossings')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')

plt.tight_layout()
plt.show()